# GeoFire-X — B1/B2 sur le corpus compact intégral
Après publication privée autorisée et vérifiée : attacher les 11 datasets listés dans dataset-sources.json. Aucune reconstruction, aucun B0 relancé. T=3, seed42, 250 train, validation 2020 par événements complets. Les scores sont comparés aux B0 existants sur le même domaine. Les campagnes sont désactivées tant que leur budget GPU n’est pas autorisé.

In [ ]:
import base64,io,zipfile,sys,json,time
from pathlib import Path
workspace=Path("/kaggle/working/compact-benchmark-"+time.strftime("%Y%m%d_%H%M%S"));workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:z.extractall(workspace)
sys.path[:0]=[str(workspace/"src"),str(workspace)]
from compact_publication import restore
expected={'ada2f8b9a3f7f340efed521b15bf3fe4e3b4151fa39eab06ca2055518f58ab58', '793223fda8a5f0d06e792bcd09b2604a97252952a5323b144184a8e4f1496a4f', '001a171f309ac58f4160269d46b9163b0c0c26c7a488cc545f6fa1b42e55d72a', '49e5551860a6d14c020c86dc33118a7cd2e0ca3dc9b470e6e95af6061817fa81', '4aa8b2818192a2cc67941f1c1bbb8a1001e144114486bb05ecaa9b504993c6fa', '906b02993f756f4b6098ac8da11ecac173f59d20ebda70c8d2bd3bb37b8f6dbd', 'ed88f93ae8329c57f34709fdf3a59e3ec3272db8238f843d4329f2cb23e293d0', 'af3015768e0174071c4d197ad31681ef20fbe0cc1de2c1084254bfef5b91393f', '3bcde37b262d5828f98153350d961e0ad771032fdedb04ff180ace536e4cb33a', 'b5caf48618be26a75a480d91401f9de0d0936607936d09556f6308c1d76c5dcd', 'ba72fbbc58f398a7bcd471d5cb21aebe4d11ed68da02caa912c2e746ccadc04d'}
maps=list(Path("/kaggle/input").rglob("mount-map.json"))
assert {json.loads(p.read_text())["identity"] for p in maps}==expected, "Partitions manquantes ou différentes"
corpus=restore([p.parent for p in maps],workspace/"corpus",verify=True)
index=corpus/"index"
from geofirex.fast_compact import FastCompactTiles
train,val=FastCompactTiles(corpus,index,"train"),FastCompactTiles(corpus,index,"val")
assert len(train)==93603 and len(val)==78771
print("Verified",len(train),len(val),"tiles; archives linked, not extracted")

Avant une campagne : profiler le chargeur compact sur cette T4 avec les partitions effectivement montées. Les débits mesurés sur GeoTIFF et sur disque local ne sont pas présentés comme un débit du compact Kaggle. Commencer par une époque B1, puis une époque B2, mêmes 32 événements val sélectionnés par métadonnées, toutes leurs tuiles sans rééquilibrage. Passer à 201 événements après mesure du temps et du scratch AP.

In [ ]:
import argparse,torch
from train_compact_benchmark import run
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
RUN_B1=False
RUN_B2=False
for model,enabled in (("B1",RUN_B1),("B2",RUN_B2)):
    if enabled:
        config=argparse.Namespace(corpus=str(corpus),index=str(index),out=str(Path("/kaggle/working")/(model+"-compact-"+time.strftime("%Y%m%d_%H%M%S"))),model=model,device="cuda",workers=4,batch_size=16,epochs=1,validation_events=32,seconds=5400,resume=None,max_updates=0,b0=str(workspace/"b0_existing.json"))
        result=run(config)
        print(json.dumps(result,indent=2))

Reprise : conserver le dernier checkpoint (toutes les 100 mises à jour et avant validation), puis définir `config.resume` sur ce fichier et une nouvelle sortie. Le curseur est celui des échantillons effectivement entraînés, pas celui des batches préchargés. Modèle, optimiseur, RNG et accumulateurs de loss sont restaurés ; changement de modèle, seed, cibles, normalisation, batch ou périmètre val refusé. Les sorties incluent loss, AP exacte, Dice/IoU/précision/rappel, PNG/NPZ réels et B0 agrégé depuis les résultats historiques. Aucun score officiel n’est publié si la validation est interrompue.